### Block 1 - Set up the API connection
Imports the necessary libraries and loads the Genius API token securely from a .env file. I also load the song dataset from Last.fm so I can collect lyrics and additional information for the same songs.

In [66]:
import warnings
warnings.filterwarnings("ignore", message="urllib3 v2 only supports OpenSSL")

import os
import re
import time
import pandas as pd
import lyricsgenius
from dotenv import load_dotenv

# Load the Genius token from .env
load_dotenv()
TOKEN = os.getenv("GENIUS_ACCESS_TOKEN")
print("Token loaded:", TOKEN is not None)

# Create the Genius client
genius = lyricsgenius.Genius(
    TOKEN,
    timeout=30,
    retries=3,
    remove_section_headers=True,   # remove [Chorus], [Verse 1], etc.
    skip_non_songs=True,
    verbose=False,
)

ARTIST = "Ariana Grande"

# The songs from Part 1; we look up each one on Genius
lastfm_df = pd.read_csv("../data/lastfm_tracks.csv")
EXPECTED_ALBUMS = list(lastfm_df["album"].unique())
print(len(lastfm_df), "songs from Last.fm:", EXPECTED_ALBUMS)

Token loaded: True
66 songs from Last.fm: ['eternal sunshine', 'Positions', 'Yours Truly', 'Sweetener', 'thank u, next']


### Block 2 - Create a function to extract lyrics
Defines a reusable function to download lyrics from Genius song pages. I want to removes unnecessary text, like verse and chorus labels, but keeping line breaks so the lyrics are clean and ready for text analysis.

In [67]:
from curl_cffi import requests as cffi_requests
from bs4 import BeautifulSoup

def get_lyrics(url):
    """Download a Genius song page and pull out the lyrics text."""
    r = cffi_requests.get(url, impersonate="chrome", timeout=30)
    r.raise_for_status()
    soup = BeautifulSoup(r.text, "html.parser")
    containers = soup.select('div[data-lyrics-container="true"]')   # where Genius puts lyrics
    if not containers:
        return None
    parts = []
    for c in containers:
        for extra in c.select('[data-exclude-from-selection="true"]'):
            extra.decompose()                  # remove page text that isn't lyrics
        for br in c.find_all("br"):
            br.replace_with("\n")              # keep line breaks
        parts.append(c.get_text())
    text = "\n".join(parts)
    text = re.sub(r"\[.*?\]", "", text)        # remove [Chorus], [Verse 1], etc.
    return text.strip()


### Block 3 - Create functions to search for songs and collect details
Defines helper functions to find Ari's songs on Genius and find their lyrics, release dates, pageviews, and songwriting credits. I want to collect consistent info for every song.

In [68]:
def find_song(title):
    """Search the official Genius API and return the first result by the artist."""
    hits = genius.search_songs(f"{title} {ARTIST}")["hits"]
    for h in hits:
        result = h["result"]
        if result["primary_artist"]["name"].lower() == ARTIST.lower():
            return result
    return None

def build_row(song_id):
    """Get one song's details (official API) and lyrics (song page) by its Genius ID."""
    song = genius.song(song_id)["song"]

    def names(key):
        """Turn a list of artist dicts into one comma-separated string."""
        return ", ".join(a["name"] for a in song.get(key) or [])

    return {
        # Required fields
        "album": (song.get("album") or {}).get("name"),
        "song_name": song["title"],
        "lyrics": get_lyrics(song["url"]),
        # ID, used to apply manual fixes
        "genius_song_id": song["id"],
        # Additional fields
        "release_date": song.get("release_date_for_display"),
        "pageviews": (song.get("stats") or {}).get("pageviews"),
        "featured_artists": names("featured_artists"),
        "writer_artists": names("writer_artists"),
        "producer_artists": names("producer_artists"),
        "annotation_count": song.get("annotation_count"),
        "pyongs_count": song.get("pyongs_count"),
    }

def show_hits(query):
    """Print the top search results, to choose a song ID by hand."""
    for h in genius.search_songs(query)["hits"][:10]:
        r = h["result"]
        print(r["id"], "|", r["full_title"])

### Block 4 - Check song matches between Last.fm and Genius
Searches for each song from the Last.fm dataset on Genius and checks whether it belongs to one of the five albums. I also need to find missing songs or incorrect matches before collecting the full dataset.

In [69]:
# Run the automatic search for every Last.fm song (no lyrics, to save time)
# and record which Genius song and album it matched
check_rows = []
for title in lastfm_df["song_name"]:
    hit = find_song(title)
    if hit is None:
        check_rows.append({"lastfm_title": title, "genius_title": None,
                           "genius_album": None, "genius_id": None})
        continue
    album = (genius.song(hit["id"])["song"].get("album") or {}).get("name")
    check_rows.append({"lastfm_title": title, "genius_title": hit["title"],
                       "genius_album": album, "genius_id": hit["id"]})
    time.sleep(0.3)

check_df = pd.DataFrame(check_rows)

# Problems: not found, or mismatched to another song
problems = check_df[check_df["genius_album"].isna() |
                    ~check_df["genius_album"].isin(EXPECTED_ALBUMS)]
problems

,lastfm_title,genius_title,genius_album,genius_id
37,Popular Song,Popular (Commentary),Wicked: The Soundtrack (Commentary),11700464


### Block 5 - Investigate incorrect song matches
Manually searches for "Popular Song" to find its correct Genius entry. Popular Song is officially credited to MIKA featuring Ari, so the automatic search may make mistakes on this.

In [70]:
# Search for the problem song by hand and choose the correct studio version
show_hits("Popular Song MIKA Ariana Grande")

5451485 | MIKA - Popular Song (Remix) (Ft. Ariana Grande) (Traducción al Español) by Genius Traducciones al Español
7546349 | MIKA - Popular Song (Remix) (Ft. Ariana Grande) [Tradução em Português] by Genius Brasil Traduções
12364684 | MIKA - Popular Song (Remix) (Ft. Ariana Grande) (Traduzione Italiana) by Genius Traduzioni Italiane
194207 | Popular Song (Remix) by MIKA (Ft. Ariana Grande)
5278573 | Polka's Popular Transcriptions by polka
5104758 | Maxenn's Transcriptions by Maxenn
12804002 | 2026 favorites (albums, eps & songs) by miloml
3694434 | Discos Traduzidos by Genius Brasil Traduções
12911177 | kyouni’s 2026 listening log by kyouni


### Block 6 - Fix incorrect song matches
Manually assigns the correct Genius ID and album to "Popular Song." I need to make sure the song is included in the dataset and grouped under the album I'm analyzing.

In [71]:
# The automatic search failed or matched the wrong version for these songs,
# so their Genius IDs were chosen by hand with show_hits().
# Keys must match the song names in the Last.fm data exactly.
MANUAL_IDS = {    
    "Popular Song": 194207,          # officially by MIKA ft. Ariana Grande, so search skipped it
}

# Songs whose Genius album should be replaced
ALBUM_OVERRIDES = {194207: "Yours Truly"}   # Popular Song is filed under MIKA on Genius

# Check that every key matches a Last.fm song name
missing_keys = [k for k in MANUAL_IDS if k not in set(lastfm_df["song_name"])]
print("Keys not found in Last.fm data:", missing_keys)

Keys not found in Last.fm data: []


### Block 7 - Collect lyrics and song information
Loops through all the songs, collecting their lyrics and additional details from Genius. Also makes sure to use manual correction when needed and one failed request won't stop the whole process.

In [72]:
rows = []
for _, row in lastfm_df.iterrows():
    title = row["song_name"]
    try:
        if title in MANUAL_IDS:                  # use the hand-picked ID
            song_id = MANUAL_IDS[title]
        else:                                    # otherwise search automatically
            hit = find_song(title)
            if hit is None:
                print("Not found:", title)
                continue
            song_id = hit["id"]
        rows.append(build_row(song_id))
    except Exception as e:                       # keep going if one song fails
        print("Failed:", title, "-", e)
    time.sleep(0.5)                              # pause between songs

genius_df = pd.DataFrame(rows)

# Apply album fixes
for song_id, album in ALBUM_OVERRIDES.items():
    genius_df.loc[genius_df["genius_song_id"] == song_id, "album"] = album

print(genius_df.shape)
genius_df["album"].value_counts(dropna=False)

(66, 11)


album
Sweetener           15
Positions           14
eternal sunshine    13
Yours Truly         12
thank u, next       12
Name: count, dtype: int64

### Block 8 - Double check the dataset for issues
Checks for incorrect albums, duplicate songs, missing values, and missing lyrics. I also preview a sample of the lyrics to make sure the text was extracted and formatted correctly before saving.

In [73]:
# 1) Songs whose album isn't one of our 5 (should be empty)
print("Songs on other albums:")
print(genius_df[~genius_df["album"].isin(EXPECTED_ALBUMS)][["album", "song_name"]])

# 2) Songs that appear more than once (should be empty)
print("\nDuplicate songs:")
print(genius_df[genius_df.duplicated("genius_song_id", keep=False)][["album", "song_name"]])

# 3) Missing values per column
print("\nMissing values:")
print(genius_df.isna().sum())

# 4) Songs with no lyrics, if any
print("\nSongs without lyrics:")
print(genius_df[genius_df["lyrics"].isna()][["album", "song_name"]])

# 5) Preview the start of one song's lyrics to check formatting
print("\nLyrics preview:")
print(genius_df["lyrics"].dropna().iloc[0][:300])

Songs on other albums:
Empty DataFrame
Columns: [album, song_name]
Index: []

Duplicate songs:
Empty DataFrame
Columns: [album, song_name]
Index: []

Missing values:
album               0
song_name           0
lyrics              0
genius_song_id      0
release_date        0
pageviews           0
featured_artists    0
writer_artists      0
producer_artists    0
annotation_count    0
pyongs_count        0
dtype: int64

Songs without lyrics:
Empty DataFrame
Columns: [album, song_name]
Index: []

Lyrics preview:
Uh


How can I tell if I'm in the right relationship?
Aren't you really s'posed to know that shit?
Feel it in your bones and own that shit? I don't know
Then I had this interaction I've been thinkin' 'bout for like five weeks
Wonder if he's thinkin' 'bout it too and smiling
Wonder if he knows that t


### Block 9 - Save the Genius dataset
Exports the collected song information and lyrics to a CSV file.

In [74]:
genius_df.to_csv("../data/genius_tracks.csv", index=False)
genius_df.head()

,album,song_name,lyrics,genius_song_id,release_date,pageviews,featured_artists,writer_artists,producer_artists,annotation_count,pyongs_count
0,eternal sunshine,intro (end of the world),Uh\n\n\nHow can I tell if I'm in the right rel...,9990963,"March 8, 2024",657441,,"Ariana Grande, Shintaro Yasuda, Nick Lee, Aaro...","Ariana Grande, Shintaro Yasuda, Nick Lee, Aaro...",3,5
1,eternal sunshine,bye,"Bye-bye\nBoy, bye, hmm\nHmm\nBye-bye\nBoy, bye...",10029453,"March 8, 2024",488608,,"Ariana Grande, Max Martin, ILYA","Ariana Grande, Max Martin, ILYA",11,3
2,eternal sunshine,don’t wanna break up again,Mm\n\n\nI fall asleep cryin'\nYou turn up the ...,10068487,"March 8, 2024",449767,,"Ariana Grande, Max Martin, ILYA","Ariana Grande, Max Martin, ILYA",13,2
3,eternal sunshine,Saturn Returns Interlude,"When we're all born, Saturn's somewhere\nAnd t...",10105322,"March 8, 2024",181339,,"Ariana Grande, Diana Garland, Max Martin, ILYA","Ariana Grande, Max Martin, ILYA, Will Loftis",6,4
4,eternal sunshine,eternal sunshine,I\n\n\nI don't care what people say\nWe both k...,9990966,"March 8, 2024",930583,,"Ariana Grande, Max Martin, Shintaro Yasuda, Da...","Max Martin, Ariana Grande, Shintaro Yasuda, Da...",21,13
